# LULC Semantic Segmentation from Sentinel-1

Training and evaluation of a land use / land cover segmentation model that uses **Sentinel-1 VV/VH backscatter only**.

- **Dataset:** 116 global sites, 2021, 12 monthly composites, 8 classes. Labels are the consensus of WorldCover and Dynamic World (see `Dataset/README.md`).
- **Model:** U-Net with an ImageNet-pretrained ResNet-34 encoder.
- **Variants compared:**
  - `single_month`: one monthly composite as input.
  - `three_month`: the target month plus the two previous months.
- **Imbalance handling:** class-weighted sparse categorical cross-entropy plus Dice loss, and a class-aware patch sampler.
- **Early stopping** on validation mIoU.

**Kaggle setup:**
1. Add the dataset (folders `s1/`, `labels/`, `patches/`) via *Add Input*.
2. Choose accelerator *GPU T4 x1* or *P100*.
3. Turn Internet on. It is needed for the pretrained weights, and for `rasterio` if it is not installed.

## 1. Setup

In [ ]:
import subprocess, sys
try:
    import rasterio
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
    import rasterio

import json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| rasterio", rasterio.__version__)

In [ ]:
def find_data_root():
    for base in [Path("/kaggle/input"), Path("Dataset"), Path("../Dataset")]:
        if base.exists():
            for hit in base.rglob("train_patches.csv"):
                return hit.parent.parent
    raise FileNotFoundError("Dataset not found: attach the dataset with folders s1/, labels/, patches/")

DATA = find_data_root()
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)
print("data:", DATA)
print("output:", OUT)

## 2. Dataset constants

- **Backscatter encoding:** stored as uint8, with `dB = code * 0.2 - 35` and code 0 = no data.
- **Labels:** 0 is *ignore* (the two label maps disagree). Classes 1-8 are trained as indices 0-7.

In [ ]:
CLASS_NAMES = ["Built-up", "Cropland", "Trees", "Shrub & grass", "Bare / sparse",
               "Water", "Wetland & mangrove", "Snow & ice"]
CLASS_COLORS = ["#e31a1c", "#ffd92f", "#1a9850", "#a6d96a", "#bf812d", "#2c7fb8", "#41b6c4", "#f0f0f0"]
N_CLASSES = len(CLASS_NAMES)
IGNORE = -1
PATCH = 256
DB_MIN, DB_STEP = -35.0, 0.2

DB_LUT = (np.arange(256, dtype=np.float32) * DB_STEP + DB_MIN)
DB_LUT[0] = np.nan

CMAP = ListedColormap(["#000000"] + CLASS_COLORS)
LEGEND = [Patch(color=c, label=n) for n, c in zip(["ignore"] + CLASS_NAMES, ["#000000"] + CLASS_COLORS)]

## 3. Load the dataset into memory

In [ ]:
SPLITS = ["train", "val", "test"]
index = {s: pd.read_csv(DATA / "patches" / f"{s}_patches.csv") for s in SPLITS}
for s in SPLITS:
    print(f"{s:5}: {len(index[s]):5} patches from {index[s].site.nunique():3} sites")

X, Y = {}, {}
t0 = time.time()
for s in SPLITS:
    df = index[s]
    X[s] = np.empty((len(df), 12, 2, PATCH, PATCH), dtype=np.uint8)
    Y[s] = np.empty((len(df), PATCH, PATCH), dtype=np.int16)
    for site, rows in df.groupby("site"):
        with rasterio.open(DATA / "s1" / f"{site}.tif") as src:
            s1 = src.read().reshape(12, 2, src.height, src.width)
        with rasterio.open(DATA / "labels" / f"{site}.tif") as src:
            lbl = src.read(1)
        for i, r, c in zip(rows.index, rows.row, rows.col):
            X[s][i] = s1[:, :, r:r + PATCH, c:c + PATCH]
            Y[s][i] = lbl[r:r + PATCH, c:c + PATCH].astype(np.int16) - 1
print(f"loaded in {time.time() - t0:.0f}s, "
      f"{sum(x.nbytes for x in X.values()) / 1e9:.2f} GB inputs")

## 4. Exploration: class distribution and samples

In [ ]:
counts = {s: np.bincount(Y[s][Y[s] >= 0].ravel(), minlength=N_CLASSES) for s in SPLITS}
dist = pd.DataFrame({s: counts[s] / counts[s].sum() * 100 for s in SPLITS}, index=CLASS_NAMES).round(1)
display(dist.rename(columns=lambda s: f"{s} %"))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
dist.plot.bar(ax=axes[0], color=["#4c72b0", "#dd8452", "#55a868"])
axes[0].set_ylabel("% of labelled pixels")
axes[0].set_title("Class distribution per split")
axes[0].tick_params(axis="x", rotation=30)
pd.crosstab(pd.concat([index[s].assign(split=s) for s in SPLITS]).region,
            pd.concat([index[s].assign(split=s) for s in SPLITS]).split).plot.bar(ax=axes[1])
axes[1].set_ylabel("patches")
axes[1].set_title("Patches per region and split")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
def false_colour(codes_2hw):
    vv, vh = DB_LUT[codes_2hw[0]], DB_LUT[codes_2hw[1]]
    rgb = np.stack([vv, vh, vv - vh], -1)
    for b, (lo, hi) in enumerate([(-20, 0), (-27, -7), (2, 15)]):
        rgb[..., b] = np.clip((rgb[..., b] - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(rgb)

rng = np.random.default_rng(0)
picks = rng.choice(len(X["train"]), 4, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for j, i in enumerate(picks):
    axes[0, j].imshow(false_colour(X["train"][i, 0]))
    axes[0, j].set_title(f"{index['train'].site[i]} - January")
    axes[1, j].imshow(false_colour(X["train"][i, 6]))
    axes[1, j].set_title("July")
    axes[2, j].imshow(Y["train"][i] + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
    axes[2, j].set_title("label")
for ax in axes.ravel():
    ax.axis("off")
fig.legend(handles=LEGEND, loc="lower center", ncol=9)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## 5. Input features and normalisation

Each month gives three channels:
- **VV** (dB)
- **VH** (dB)
- **VV - VH**, the cross-polarisation ratio in dB. It separates volume scattering (vegetation) from surface and double-bounce scattering.

Channel means and standard deviations come from the **train split only**. Missing pixels are set to 0 after standardisation, which is the channel mean.

In [ ]:
sample = DB_LUT[X["train"][::4, :, :, ::4, ::4]]
feats = np.stack([sample[:, :, 0], sample[:, :, 1], sample[:, :, 0] - sample[:, :, 1]], 0).reshape(3, -1)
CH_MEAN = np.nanmean(feats, axis=1).astype(np.float32)
CH_STD = np.nanstd(feats, axis=1).astype(np.float32)
del sample, feats
print("channel mean (VV, VH, VV-VH):", CH_MEAN.round(3))
print("channel std  (VV, VH, VV-VH):", CH_STD.round(3))


def window_months(month, n_months):
    return [max(0, month - j) for j in range(n_months - 1, -1, -1)]


def to_features(codes):
    db = DB_LUT[codes]
    vv, vh = db[:, 0], db[:, 1]
    f = np.stack([vv, vh, vv - vh], 1)
    f = (f - CH_MEAN[None, :, None, None]) / CH_STD[None, :, None, None]
    return np.nan_to_num(f, nan=0.0).reshape(-1, *codes.shape[-2:]).astype(np.float32)

## 6. PyTorch datasets, augmentation and imbalance-aware sampling

**Training items**
- Each training item is a patch plus a **random month**. Every month shares the same annual label, so the model learns to classify any single month, which is what the prototype requires.
- With `n_months > 1`, the input also contains the previous months. For January and February, the earliest available month is repeated.

**Augmentation:** random horizontal and vertical flips, and 90-degree rotations.

**Imbalance handling**
1. **Class weights.** The loss uses class weights `w_c = sqrt(median_freq / freq_c)`, normalised to mean 1. The square root damps the weights so that rare classes don't dominate.
2. **Patch sampler.** A `WeightedRandomSampler` draws a patch with probability proportional to the largest class weight among the classes covering at least 1% of it. Patches containing rare classes are therefore seen more often.

**Evaluation:** val and test use every patch with **all 12 months**.

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, x, y, n_months=1, train=False):
        self.x, self.y, self.n_months, self.train = x, y, n_months, train
        self.items = list(range(len(x))) if train else [(i, m) for i in range(len(x)) for m in range(12)]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        if self.train:
            i, month = self.items[k], np.random.randint(12)
        else:
            i, month = self.items[k]
        feats = to_features(self.x[i, window_months(month, self.n_months)])
        label = self.y[i]
        if self.train:
            if np.random.rand() < 0.5:
                feats, label = feats[:, :, ::-1], label[:, ::-1]
            if np.random.rand() < 0.5:
                feats, label = feats[:, ::-1, :], label[::-1, :]
            rot = np.random.randint(4)
            feats, label = np.rot90(feats, rot, (1, 2)), np.rot90(label, rot)
        return torch.from_numpy(feats.copy()), torch.from_numpy(label.astype(np.int64)), month


def seed_worker(worker_id):
    np.random.seed(torch.initial_seed() % 2 ** 32)


freq = counts["train"] / counts["train"].sum()
CLASS_WEIGHTS = np.sqrt(np.median(freq) / np.clip(freq, 1e-6, None))
CLASS_WEIGHTS = (CLASS_WEIGHTS / CLASS_WEIGHTS.mean()).astype(np.float32)
display(pd.DataFrame({"train share %": (freq * 100).round(2), "loss weight": CLASS_WEIGHTS.round(2)},
                     index=CLASS_NAMES))

patch_frac = np.stack([np.bincount(y[y >= 0], minlength=N_CLASSES) / max((y >= 0).sum(), 1)
                       for y in Y["train"]])
PATCH_WEIGHTS = np.where(patch_frac >= 0.01, CLASS_WEIGHTS[None], 0).max(1)
print("patch sampling weight range:", PATCH_WEIGHTS.min().round(2), "-", PATCH_WEIGHTS.max().round(2))

## 7. Model: U-Net with ResNet-34 encoder

**Encoder**
- ImageNet-pretrained ResNet-34 from torchvision, with 5 scales: /2, /4, /8, /16 and /32.
- Pretrained filters (edges, textures) transfer well to SAR texture even though the input is not RGB.

**First convolution**
- Its RGB filters are averaged and copied to the `3 x n_months` input channels.
- They are scaled by `3 / in_channels`, which keeps the activation scale the same.

**Decoder**
- 5 upsampling blocks: bilinear x2, concatenate the encoder skip connection, then 2 x (conv3x3 + BatchNorm + ReLU).
- Followed by a 1x1 classification head with 8 outputs.

**Why U-Net**
- The skip connections restore the 10 m detail needed for roads, rivers and field edges.
- The deep encoder gives about 2.5 km of context for land-use texture.
- At about 24 M parameters, it trains on one Kaggle GPU in well under an hour.

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], 1)
        return self.conv(x)


class UNetResNet34(nn.Module):
    def __init__(self, in_ch, n_classes, pretrained=True):
        super().__init__()
        try:
            enc = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)
        except Exception as err:
            print("pretrained weights unavailable, using random init:", err)
            enc = torchvision.models.resnet34(weights=None)
        conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            conv1.weight.copy_(enc.conv1.weight.mean(1, keepdim=True).repeat(1, in_ch, 1, 1) * 3 / in_ch)
        self.stem = nn.Sequential(conv1, enc.bn1, enc.relu)
        self.pool = enc.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = enc.layer1, enc.layer2, enc.layer3, enc.layer4
        self.dec4 = UpBlock(512, 256, 256)
        self.dec3 = UpBlock(256, 128, 128)
        self.dec2 = UpBlock(128, 64, 64)
        self.dec1 = UpBlock(64, 64, 64)
        self.dec0 = UpBlock(64, 0, 32)
        self.dropout = nn.Dropout2d(0.1)
        self.head = nn.Conv2d(32, n_classes, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.enc1(self.pool(s0))
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        d = self.dec4(s4, s3)
        d = self.dec3(d, s2)
        d = self.dec2(d, s1)
        d = self.dec1(d, s0)
        d = self.dec0(d)
        return self.head(self.dropout(d))


m = UNetResNet34(3, N_CLASSES, pretrained=False)
print(f"parameters: {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M")
print("output shape:", tuple(m(torch.zeros(1, 3, PATCH, PATCH)).shape))
del m

## 8. Loss and metrics

**Loss = weighted sparse categorical cross-entropy + 0.5 x Dice**
- *Sparse categorical cross-entropy:* the targets are integer class indices, not one-hot vectors. This is PyTorch's `CrossEntropyLoss` with integer targets, which is the same as Keras' `SparseCategoricalCrossentropy`. Ignored pixels (`-1`) are excluded with `ignore_index`.
- *Dice loss:* the mean over the classes present in the batch. It optimises overlap directly and is insensitive to class frequency, which complements the weighted cross-entropy and lifts the IoU of rare classes.

**Metrics** are computed from a confusion matrix over all labelled pixels:
- overall accuracy (OA)
- per-class IoU and F1
- mIoU: the mean IoU over the 8 classes

In [ ]:
class CEDiceLoss(nn.Module):
    def __init__(self, class_weights, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE)
        self.dice_weight = dice_weight

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        valid = (target != IGNORE).unsqueeze(1).float()
        probs = logits.float().softmax(1) * valid
        onehot = F.one_hot(target.clamp(min=0), N_CLASSES).permute(0, 3, 1, 2).float() * valid
        inter = (probs * onehot).sum((0, 2, 3))
        denom = (probs + onehot).sum((0, 2, 3))
        present = onehot.sum((0, 2, 3)) > 0
        dice = 1 - (2 * inter + 1) / (denom + 1)
        dice = dice[present].mean() if present.any() else ce * 0
        return ce + self.dice_weight * dice


def update_confusion(conf, logits, target):
    pred = logits.argmax(1)
    valid = target != IGNORE
    conf += torch.bincount(target[valid] * N_CLASSES + pred[valid],
                           minlength=N_CLASSES ** 2).reshape(N_CLASSES, N_CLASSES)
    return conf


def metrics_from_confusion(conf):
    conf = conf.double()
    tp = conf.diag()
    fp, fn = conf.sum(0) - tp, conf.sum(1) - tp
    iou = tp / (tp + fp + fn).clamp(min=1)
    f1 = 2 * tp / (2 * tp + fp + fn).clamp(min=1)
    present = conf.sum(1) > 0
    return {"OA": (tp.sum() / conf.sum()).item(),
            "mIoU": iou[present].mean().item(),
            "IoU": iou.cpu().numpy(), "F1": f1.cpu().numpy()}

## 9. Training loop with early stopping

**Optimiser and schedule**
- AdamW (lr 3e-4, weight decay 1e-4) with mixed precision.
- `ReduceLROnPlateau` halves the learning rate when val mIoU stalls for 3 epochs.

**Epochs**
- One epoch = 2 x the number of train patches, drawn by the class-aware sampler. Each draw gets a random month.

**Early stopping**
- Training stops when val mIoU has not improved for `PATIENCE` epochs.
- The best checkpoint (highest val mIoU) is restored and saved.

In [ ]:
MAX_EPOCHS = 60
PATIENCE = 8
BATCH = 16
NUM_WORKERS = 2


def evaluate(model, loader, per_month=False):
    model.eval()
    conf = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE)
    month_conf = torch.zeros(12, N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE) if per_month else None
    loss_sum, n = 0.0, 0
    with torch.no_grad(), torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
        for xb, yb, mb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            loss_sum += criterion(logits, yb).item() * len(xb)
            n += len(xb)
            update_confusion(conf, logits, yb)
            if per_month:
                for m in mb.unique():
                    sel = mb.to(DEVICE) == m
                    update_confusion(month_conf[m], logits[sel], yb[sel])
    out = metrics_from_confusion(conf)
    out["loss"], out["confusion"] = loss_sum / n, conf.cpu().numpy()
    if per_month:
        out["month_mIoU"] = [metrics_from_confusion(month_conf[m])["mIoU"] for m in range(12)]
    return out


def train_variant(name, n_months):
    print(f"=== {name}: {n_months} month(s) -> {3 * n_months} input channels ===")
    train_ds = PatchDataset(X["train"], Y["train"], n_months, train=True)
    val_ds = PatchDataset(X["val"], Y["val"], n_months)
    sampler = WeightedRandomSampler(torch.as_tensor(PATCH_WEIGHTS, dtype=torch.double),
                                    num_samples=2 * len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, BATCH, sampler=sampler, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True, persistent_workers=NUM_WORKERS > 0,
                          worker_init_fn=seed_worker)
    val_dl = DataLoader(val_ds, 2 * BATCH, num_workers=NUM_WORKERS, pin_memory=True)

    model = UNetResNet34(3 * n_months, N_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

    history, best, best_state, wait = [], -1.0, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0, loss_sum, n = time.time(), 0.0, 0
        conf = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long, device=DEVICE)
        for xb, yb, _ in train_dl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(xb)
            n += len(xb)
            update_confusion(conf, logits.detach(), yb)
        train_m = metrics_from_confusion(conf)
        val_m = evaluate(model, val_dl)
        scheduler.step(val_m["mIoU"])
        history.append({"epoch": epoch, "train_loss": loss_sum / n, "val_loss": val_m["loss"],
                        "train_OA": train_m["OA"], "val_OA": val_m["OA"],
                        "train_mIoU": train_m["mIoU"], "val_mIoU": val_m["mIoU"],
                        "lr": optimizer.param_groups[0]["lr"]})
        improved = val_m["mIoU"] > best
        if improved:
            best, wait = val_m["mIoU"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        print(f"epoch {epoch:2d} | train loss {loss_sum / n:.3f} mIoU {train_m['mIoU']:.3f} | "
              f"val loss {val_m['loss']:.3f} OA {val_m['OA']:.3f} mIoU {val_m['mIoU']:.3f} | "
              f"lr {optimizer.param_groups[0]['lr']:.1e} | {time.time() - t0:.0f}s{' *' if improved else ''}")
        if wait >= PATIENCE:
            print(f"early stopping: no val mIoU improvement for {PATIENCE} epochs")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, OUT / f"{name}_best.pt")
    pd.DataFrame(history).to_csv(OUT / f"{name}_history.csv", index=False)
    print(f"best val mIoU {best:.3f}, weights saved to {OUT / f'{name}_best.pt'}")
    return model, pd.DataFrame(history)


criterion = CEDiceLoss(torch.tensor(CLASS_WEIGHTS, device=DEVICE))
VARIANTS = {"single_month": 1, "three_month": 3}

In [ ]:
models, histories = {}, {}
for name, n_months in VARIANTS.items():
    models[name], histories[name] = train_variant(name, n_months)

## 10. Training curves

In [ ]:
fig, axes = plt.subplots(len(histories), 3, figsize=(18, 4.5 * len(histories)), squeeze=False)
for row, (name, h) in enumerate(histories.items()):
    for ax, metric in zip(axes[row], ["loss", "OA", "mIoU"]):
        ax.plot(h.epoch, h[f"train_{metric}"], label="train")
        ax.plot(h.epoch, h[f"val_{metric}"], label="val")
        best_epoch = h.val_mIoU.idxmax()
        ax.axvline(h.epoch[best_epoch], color="grey", ls="--", lw=1, label="best (early stopping)")
        ax.set_title(f"{name}: {metric}")
        ax.set_xlabel("epoch")
        ax.legend()
plt.tight_layout()
plt.savefig(OUT / "training_curves.png", dpi=120)
plt.show()

## 11. Test set evaluation

The test set has 20 sites the model has never seen, spread over all regions. Every test patch is evaluated with all 12 monthly composites. The reported metrics are:
- **Overall:** OA and mIoU.
- **Per class:** IoU and F1.
- **Variant comparison** between `single_month` and `three_month`.
- **Breakdowns:** per region, per site type, and per month. The per-month view shows seasonal effects.

In [ ]:
results = {}
for name, n_months in VARIANTS.items():
    test_dl = DataLoader(PatchDataset(X["test"], Y["test"], n_months), 2 * BATCH,
                         num_workers=NUM_WORKERS, pin_memory=True)
    results[name] = evaluate(models[name], test_dl, per_month=True)

comparison = pd.DataFrame({name: {"OA": r["OA"], "mIoU": r["mIoU"],
                                  "best val mIoU": histories[name].val_mIoU.max(),
                                  "epochs": len(histories[name])}
                           for name, r in results.items()}).T
display(comparison.round(4))

per_class = pd.concat({name: pd.DataFrame({"IoU": r["IoU"], "F1": r["F1"]}, index=CLASS_NAMES)
                       for name, r in results.items()}, axis=1)
display(per_class.round(3))

BEST = comparison.mIoU.idxmax()
print("best variant on test:", BEST)
json.dump({name: {"OA": r["OA"], "mIoU": r["mIoU"],
                  "IoU": dict(zip(CLASS_NAMES, r["IoU"].round(4).tolist())),
                  "F1": dict(zip(CLASS_NAMES, r["F1"].round(4).tolist())),
                  "month_mIoU": [round(v, 4) for v in r["month_mIoU"]]}
           for name, r in results.items()}, open(OUT / "test_metrics.json", "w"), indent=2)

In [ ]:
conf = results[BEST]["confusion"].astype(np.float64)
conf_norm = conf / conf.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(conf_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f"{conf_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if conf_norm[i, j] > 0.5 else "black", fontsize=8)
ax.set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=40, ha="right")
ax.set_yticks(range(N_CLASSES), CLASS_NAMES)
ax.set_xlabel("predicted")
ax.set_ylabel("reference")
ax.set_title(f"Test confusion matrix (row-normalised), {BEST}")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
def group_scores(model, n_months, column):
    rows = []
    for group, df in index["test"].groupby(column):
        ids = df.index.to_numpy()
        dl = DataLoader(PatchDataset(X["test"][ids], Y["test"][ids], n_months), 2 * BATCH)
        r = evaluate(model, dl)
        rows.append({column: group, "patches": len(ids), "OA": r["OA"], "mIoU": r["mIoU"]})
    return pd.DataFrame(rows).set_index(column).round(3)

display(group_scores(models[BEST], VARIANTS[BEST], "region"))
display(group_scores(models[BEST], VARIANTS[BEST], "focus"))

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
plt.figure(figsize=(10, 4))
for name, r in results.items():
    plt.plot(months, r["month_mIoU"], marker="o", label=name)
plt.ylabel("test mIoU")
plt.title("Test mIoU per input month (seasonal effect)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "month_miou.png", dpi=120)
plt.show()

## 12. Example predictions on unseen test sites

In [ ]:
model = models[BEST].eval()
picks = [index["test"][index["test"].focus == f].index[0] for f in index["test"].focus.unique()][:6]
fig, axes = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)))
for row, i in enumerate(picks):
    month = 6
    x = to_features(X["test"][i, window_months(month, VARIANTS[BEST])])
    with torch.no_grad():
        pred = model(torch.from_numpy(x)[None].to(DEVICE)).argmax(1)[0].cpu().numpy()
    axes[row, 0].imshow(false_colour(X["test"][i, month]))
    axes[row, 0].set_title(f"{index['test'].site[i]} ({index['test'].focus[i]}), July S1")
    axes[row, 1].imshow(Y["test"][i] + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
    axes[row, 1].set_title("reference (black = ignore)")
    axes[row, 2].imshow(pred + 1, cmap=CMAP, vmin=0, vmax=N_CLASSES, interpolation="nearest")
    axes[row, 2].set_title("prediction")
for ax in axes.ravel():
    ax.axis("off")
fig.legend(handles=LEGEND, loc="lower center", ncol=9)
plt.tight_layout(rect=(0, 0.02, 1, 1))
plt.savefig(OUT / "example_predictions.png", dpi=110)
plt.show()

## 13. Export for the prototype

The best model's weights are saved together with everything the prototype needs to reproduce the preprocessing:
- the input channels, the number of months and the window rule;
- the normalisation statistics;
- the uint8 decoding;
- the class names and colours.

In [ ]:
model_card = {
    "variant": BEST,
    "n_months": VARIANTS[BEST],
    "channels_per_month": ["VV_dB", "VH_dB", "VV_minus_VH_dB"],
    "month_window": "target month and previous months, oldest first; earliest month repeated at the start",
    "channel_mean": CH_MEAN.tolist(),
    "channel_std": CH_STD.tolist(),
    "nan_fill_after_standardisation": 0.0,
    "class_names": CLASS_NAMES,
    "class_colors": CLASS_COLORS,
    "patch_size": PATCH,
    "architecture": "UNetResNet34 (this notebook)",
    "weights": f"{BEST}_best.pt",
    "test_OA": results[BEST]["OA"],
    "test_mIoU": results[BEST]["mIoU"],
}
json.dump(model_card, open(OUT / "model_card.json", "w"), indent=2)
print(json.dumps(model_card, indent=2))
print(sorted(p.name for p in OUT.iterdir()))